# RAG Across Three Real Economics Corpora

Learning objective: use the same RAG retrieval steps on three real economics corpora that come from official source documents.


In [ ]:
from pathlib import Path  # This class helps us work with folders and file paths.
import matplotlib.pyplot as plt  # This library draws simple graphs.
from bs4 import BeautifulSoup  # This class reads HTML pages and helps us find the article text.
from pypdf import PdfReader  # This class reads text from PDF files.
from sentence_transformers import SentenceTransformer  # This class turns text into embedding vectors.
from sklearn.metrics.pairwise import cosine_similarity  # This function compares vectors with cosine similarity.


## What changed in this notebook

This notebook now uses real source files instead of short demo sentences typed by hand.

The files are stored in a local `corpora` folder, so you do not need to fetch the documents again while you study the notebook.

The three corpora are:
1. Federal Reserve monetary policy statements
2. Census and HUD new residential sales releases
3. Federal Reserve speeches about labor markets

The method stays the same in every section:
- load the documents
- chunk the documents
- embed the chunks
- retrieve the most relevant chunks
- build an answer from the retrieved evidence


In [ ]:
project_folder = Path.cwd()
corpus_folder = project_folder / 'corpora'

embedding_model_name = 'all-MiniLM-L6-v2'
embedding_model = SentenceTransformer(embedding_model_name)

common_chunk_size = 550
common_chunk_overlap = 120

print('Working folder:')
print(project_folder)
print()
print('Corpus folder:')
print(corpus_folder)
print()
print('Loaded embedding model:')
print(embedding_model_name)
print()
print('Chunk size:')
print(common_chunk_size)
print('Chunk overlap:')
print(common_chunk_overlap)


## Section 1: Monetary policy corpus

This corpus uses real Federal Reserve monetary policy statements from 2025.

A statement is the short official text that explains the committee's decision and its reasoning.
We will ask a question about why the policy rate stayed unchanged.


In [ ]:
monetary_source_names = []
monetary_source_names.append('FOMC statement on January 29, 2025')
monetary_source_names.append('FOMC statement on March 19, 2025')
monetary_source_names.append('FOMC statement on May 7, 2025')

monetary_source_urls = []
monetary_source_urls.append('https://www.federalreserve.gov/newsevents/pressreleases/monetary20250129a.htm')
monetary_source_urls.append('https://www.federalreserve.gov/newsevents/pressreleases/monetary20250319a.htm')
monetary_source_urls.append('https://www.federalreserve.gov/newsevents/pressreleases/monetary20250507a.htm')

monetary_file_paths = []
monetary_file_paths.append(corpus_folder / 'monetary' / 'fomc_2025_01_29_statement.html')
monetary_file_paths.append(corpus_folder / 'monetary' / 'fomc_2025_03_19_statement.html')
monetary_file_paths.append(corpus_folder / 'monetary' / 'fomc_2025_05_07_statement.html')

monetary_documents = []
monetary_document_index = 0

for monetary_file_path in monetary_file_paths:
    monetary_html_text = monetary_file_path.read_text(encoding='utf-8', errors='ignore')
    monetary_soup = BeautifulSoup(monetary_html_text, 'html.parser')
    monetary_article = monetary_soup.find(id='article')
    monetary_document_text = ''

    if monetary_article is not None:
        for monetary_paragraph in monetary_article.find_all('p'):
            monetary_paragraph_text = monetary_paragraph.get_text(' ', strip=True)
            if monetary_paragraph_text != '':
                if monetary_paragraph_text != 'Share':
                    if 'For media inquiries' not in monetary_paragraph_text:
                        if 'Implementation Note issued' not in monetary_paragraph_text:
                            if 'PDF' not in monetary_paragraph_text:
                                monetary_paragraph_text = monetary_paragraph_text.replace(' Share', '')
                                monetary_document_text = monetary_document_text + monetary_paragraph_text + '\n'

    monetary_documents.append(monetary_document_text.strip())

    print('Document number:', monetary_document_index)
    print('Document name:', monetary_source_names[monetary_document_index])
    print('Source URL:', monetary_source_urls[monetary_document_index])
    print('Local file:', monetary_file_paths[monetary_document_index])
    print('Character count:', len(monetary_documents[monetary_document_index]))
    print('Preview:')
    print(monetary_documents[monetary_document_index][0:500])
    print()

    monetary_document_index = monetary_document_index + 1

monetary_question = 'Why did the Committee keep the federal funds rate unchanged in these 2025 statements?'

print('Question:')
print(monetary_question)


## Chunk and retrieve from the monetary policy corpus

A chunk is one smaller piece of a longer document.

We split the statements into chunks because the retriever usually works better on shorter pieces of text than on one very long document.
Then we compare each chunk with the question.


In [ ]:
monetary_chunk_size = common_chunk_size
monetary_chunk_overlap = common_chunk_overlap
monetary_chunk_records = []
monetary_document_index = 0

for monetary_document_text in monetary_documents:
    monetary_start_position = 0
    monetary_chunk_number = 0
    monetary_text_length = len(monetary_document_text)

    while monetary_start_position < monetary_text_length:
        monetary_end_position = monetary_start_position + monetary_chunk_size

        if monetary_end_position > monetary_text_length:
            monetary_end_position = monetary_text_length

        if monetary_end_position < monetary_text_length:
            monetary_search_position = monetary_end_position

            while monetary_search_position > monetary_start_position:
                if monetary_document_text[monetary_search_position - 1].isspace() is True:
                    monetary_end_position = monetary_search_position - 1
                    break

                monetary_search_position = monetary_search_position - 1

        monetary_chunk_text = monetary_document_text[monetary_start_position:monetary_end_position].strip()

        if monetary_chunk_text != '':
            monetary_chunk_record = {
                'source_name': monetary_source_names[monetary_document_index],
                'source_url': monetary_source_urls[monetary_document_index],
                'chunk_number': monetary_chunk_number,
                'text': monetary_chunk_text,
            }
            monetary_chunk_records.append(monetary_chunk_record)
            monetary_chunk_number = monetary_chunk_number + 1

        if monetary_end_position == monetary_text_length:
            monetary_next_start_position = monetary_text_length
        else:
            monetary_next_start_position = monetary_end_position - monetary_chunk_overlap

            if monetary_next_start_position <= monetary_start_position:
                monetary_next_start_position = monetary_end_position

            while monetary_next_start_position > 0:
                if monetary_document_text[monetary_next_start_position - 1].isspace() is True:
                    break

                monetary_next_start_position = monetary_next_start_position - 1

            while monetary_next_start_position < monetary_text_length:
                if monetary_document_text[monetary_next_start_position].isspace() is False:
                    break

                monetary_next_start_position = monetary_next_start_position + 1

        monetary_start_position = monetary_next_start_position

    monetary_document_index = monetary_document_index + 1

monetary_chunk_texts = []

for monetary_chunk_record in monetary_chunk_records:
    monetary_chunk_texts.append(monetary_chunk_record['text'])

monetary_chunk_embeddings = embedding_model.encode(monetary_chunk_texts)

monetary_question_embedding = embedding_model.encode(monetary_question)
monetary_similarity_scores = []

for monetary_chunk_embedding in monetary_chunk_embeddings:
    monetary_similarity_value = cosine_similarity([monetary_question_embedding], [monetary_chunk_embedding])[0][0]
    monetary_similarity_scores.append(monetary_similarity_value)

monetary_chunk_labels = []

for monetary_chunk_record in monetary_chunk_records:
    monetary_chunk_label = monetary_chunk_record['source_name'] + '_chunk_' + str(monetary_chunk_record['chunk_number'])
    monetary_chunk_labels.append(monetary_chunk_label)

figure = plt.figure(figsize=(11, 3.5))
plt.bar(monetary_chunk_labels, monetary_similarity_scores)
plt.title('Monetary policy retrieval scores')
plt.xlabel('Chunk')
plt.ylabel('Cosine similarity')
plt.xticks(rotation=60, ha='right')
plt.tight_layout()
plt.show()
plt.close(figure)

monetary_ranked_results = []
monetary_chunk_index = 0

for monetary_similarity_value in monetary_similarity_scores:
    monetary_ranked_result = (
        monetary_similarity_value,
        monetary_chunk_records[monetary_chunk_index]['source_name'],
        monetary_chunk_records[monetary_chunk_index]['source_url'],
        monetary_chunk_records[monetary_chunk_index]['chunk_number'],
        monetary_chunk_records[monetary_chunk_index]['text'],
    )
    monetary_ranked_results.append(monetary_ranked_result)
    monetary_chunk_index = monetary_chunk_index + 1

monetary_ranked_results = sorted(monetary_ranked_results, reverse=True)

monetary_rank_number = 1

for monetary_ranked_result in monetary_ranked_results:
    if monetary_rank_number <= 3:
        print('Rank:', monetary_rank_number)
        print('Score:', round(monetary_ranked_result[0], 4))
        print('Source:', monetary_ranked_result[1])
        print('Source URL:', monetary_ranked_result[2])
        print('Chunk number:', monetary_ranked_result[3])
        print('Chunk text:')
        print(monetary_ranked_result[4])
        print()

    monetary_rank_number = monetary_rank_number + 1


## Build the monetary policy context and answer

The context is the text we hand to the answer stage.

To keep the logic simple, this notebook builds a short evidence-based answer by reusing the strongest sentences from the top retrieved chunks.
That keeps the answer grounded in the corpus.


In [ ]:
monetary_context_text = ''
monetary_top_results = []
monetary_rank_number = 1

for monetary_ranked_result in monetary_ranked_results:
    if monetary_rank_number <= 3:
        monetary_top_results.append(monetary_ranked_result)
        monetary_context_text = monetary_context_text + '[' + monetary_ranked_result[1] + ']\n'
        monetary_context_text = monetary_context_text + monetary_ranked_result[2] + '\n'
        monetary_context_text = monetary_context_text + monetary_ranked_result[4] + '\n\n'

    monetary_rank_number = monetary_rank_number + 1

print('Retrieved context:')
print(monetary_context_text)
print()

monetary_answer_keywords = []
monetary_answer_keywords.append('maintain the target range')
monetary_answer_keywords.append('balance of risks')
monetary_answer_keywords.append('inflation')
monetary_answer_keywords.append('higher unemployment')
monetary_answer_keywords.append('higher inflation')

monetary_answer_sentences = []

for monetary_top_result in monetary_top_results:
    monetary_chunk_text = monetary_top_result[4].replace('\n', ' ')
    monetary_sentence_pieces = monetary_chunk_text.split('. ')

    for monetary_sentence_piece in monetary_sentence_pieces:
        monetary_sentence = monetary_sentence_piece.strip()

        if monetary_sentence != '':
            monetary_sentence_lower = monetary_sentence.lower()

            for monetary_answer_keyword in monetary_answer_keywords:
                if monetary_answer_keyword in monetary_sentence_lower:
                    if monetary_sentence.endswith('.') is False:
                        monetary_sentence = monetary_sentence + '.'

                    if monetary_sentence not in monetary_answer_sentences:
                        monetary_answer_sentences.append(monetary_sentence)

                    break

monetary_evidence_answer = ''

for monetary_sentence in monetary_answer_sentences:
    if monetary_sentence not in monetary_evidence_answer:
        if monetary_evidence_answer != '':
            monetary_evidence_answer = monetary_evidence_answer + ' '
        monetary_evidence_answer = monetary_evidence_answer + monetary_sentence

print('Evidence-based answer:')
print(monetary_evidence_answer)


## Section 2: Housing market corpus

This corpus uses real Census and HUD new residential sales releases from early 2025.

These are PDF reports.
We will read the first page of each report because that page contains the main summary numbers and short written explanation.


In [ ]:
housing_source_names = []
housing_source_names.append('New residential sales release for January 2025')
housing_source_names.append('New residential sales release for February 2025')
housing_source_names.append('New residential sales release for March 2025')

housing_source_urls = []
housing_source_urls.append('https://www.census.gov/construction/nrs/pdf/newressales_202501.pdf')
housing_source_urls.append('https://www.census.gov/construction/nrs/pdf/newressales_202502.pdf')
housing_source_urls.append('https://www.census.gov/construction/nrs/pdf/newressales_202503.pdf')

housing_file_paths = []
housing_file_paths.append(corpus_folder / 'housing' / 'new_residential_sales_2025_01.pdf')
housing_file_paths.append(corpus_folder / 'housing' / 'new_residential_sales_2025_02.pdf')
housing_file_paths.append(corpus_folder / 'housing' / 'new_residential_sales_2025_03.pdf')

housing_documents = []
housing_document_index = 0

for housing_file_path in housing_file_paths:
    housing_reader = PdfReader(str(housing_file_path))
    housing_document_text = ''

    if len(housing_reader.pages) > 0:
        housing_first_page_text = housing_reader.pages[0].extract_text()

        if housing_first_page_text is not None:
            housing_lines = housing_first_page_text.splitlines()
            housing_started_text = False
            housing_reached_notes = False

            for housing_line in housing_lines:
                housing_clean_line = housing_line.strip()

                if 'MONTHLY NEW RESIDENTIAL SALES' in housing_clean_line:
                    housing_started_text = True

                if 'EXPLANATORY NOTES' in housing_clean_line:
                    housing_reached_notes = True

                if housing_started_text is True and housing_reached_notes is False:
                    if housing_clean_line != '':
                        housing_document_text = housing_document_text + housing_clean_line + ' '

    housing_documents.append(housing_document_text.strip())

    print('Document number:', housing_document_index)
    print('Document name:', housing_source_names[housing_document_index])
    print('Source URL:', housing_source_urls[housing_document_index])
    print('Local file:', housing_file_paths[housing_document_index])
    print('Character count:', len(housing_documents[housing_document_index]))
    print('Preview:')
    print(housing_documents[housing_document_index][0:500])
    print()

    housing_document_index = housing_document_index + 1

housing_question = 'What do these early 2025 housing releases say about prices and the number of new houses for sale?'

print('Question:')
print(housing_question)


## Chunk and retrieve from the housing market corpus

The retrieval steps are the same even though the documents came from PDFs instead of HTML pages.

That is an important RAG idea.
The same retrieval workflow can move across different file types.


In [ ]:
housing_chunk_size = common_chunk_size
housing_chunk_overlap = common_chunk_overlap
housing_chunk_records = []
housing_document_index = 0

for housing_document_text in housing_documents:
    housing_start_position = 0
    housing_chunk_number = 0
    housing_text_length = len(housing_document_text)

    while housing_start_position < housing_text_length:
        housing_end_position = housing_start_position + housing_chunk_size

        if housing_end_position > housing_text_length:
            housing_end_position = housing_text_length

        if housing_end_position < housing_text_length:
            housing_search_position = housing_end_position

            while housing_search_position > housing_start_position:
                if housing_document_text[housing_search_position - 1].isspace() is True:
                    housing_end_position = housing_search_position - 1
                    break

                housing_search_position = housing_search_position - 1

        housing_chunk_text = housing_document_text[housing_start_position:housing_end_position].strip()

        if housing_chunk_text != '':
            housing_chunk_record = {
                'source_name': housing_source_names[housing_document_index],
                'source_url': housing_source_urls[housing_document_index],
                'chunk_number': housing_chunk_number,
                'text': housing_chunk_text,
            }
            housing_chunk_records.append(housing_chunk_record)
            housing_chunk_number = housing_chunk_number + 1

        if housing_end_position == housing_text_length:
            housing_next_start_position = housing_text_length
        else:
            housing_next_start_position = housing_end_position - housing_chunk_overlap

            if housing_next_start_position <= housing_start_position:
                housing_next_start_position = housing_end_position

            while housing_next_start_position > 0:
                if housing_document_text[housing_next_start_position - 1].isspace() is True:
                    break

                housing_next_start_position = housing_next_start_position - 1

            while housing_next_start_position < housing_text_length:
                if housing_document_text[housing_next_start_position].isspace() is False:
                    break

                housing_next_start_position = housing_next_start_position + 1

        housing_start_position = housing_next_start_position

    housing_document_index = housing_document_index + 1

housing_chunk_texts = []

for housing_chunk_record in housing_chunk_records:
    housing_chunk_texts.append(housing_chunk_record['text'])

housing_chunk_embeddings = embedding_model.encode(housing_chunk_texts)

housing_question_embedding = embedding_model.encode(housing_question)
housing_similarity_scores = []

for housing_chunk_embedding in housing_chunk_embeddings:
    housing_similarity_value = cosine_similarity([housing_question_embedding], [housing_chunk_embedding])[0][0]
    housing_similarity_scores.append(housing_similarity_value)

housing_chunk_labels = []

for housing_chunk_record in housing_chunk_records:
    housing_chunk_label = housing_chunk_record['source_name'] + '_chunk_' + str(housing_chunk_record['chunk_number'])
    housing_chunk_labels.append(housing_chunk_label)

figure = plt.figure(figsize=(11, 3.5))
plt.bar(housing_chunk_labels, housing_similarity_scores)
plt.title('Housing market retrieval scores')
plt.xlabel('Chunk')
plt.ylabel('Cosine similarity')
plt.xticks(rotation=60, ha='right')
plt.tight_layout()
plt.show()
plt.close(figure)

housing_ranked_results = []
housing_chunk_index = 0

for housing_similarity_value in housing_similarity_scores:
    housing_ranked_result = (
        housing_similarity_value,
        housing_chunk_records[housing_chunk_index]['source_name'],
        housing_chunk_records[housing_chunk_index]['source_url'],
        housing_chunk_records[housing_chunk_index]['chunk_number'],
        housing_chunk_records[housing_chunk_index]['text'],
    )
    housing_ranked_results.append(housing_ranked_result)
    housing_chunk_index = housing_chunk_index + 1

housing_ranked_results = sorted(housing_ranked_results, reverse=True)

housing_rank_number = 1

for housing_ranked_result in housing_ranked_results:
    if housing_rank_number <= 3:
        print('Rank:', housing_rank_number)
        print('Score:', round(housing_ranked_result[0], 4))
        print('Source:', housing_ranked_result[1])
        print('Source URL:', housing_ranked_result[2])
        print('Chunk number:', housing_ranked_result[3])
        print('Chunk text:')
        print(housing_ranked_result[4])
        print()

    housing_rank_number = housing_rank_number + 1


## Build the housing market context and answer

The question asks about prices and houses for sale, so we expect the best chunks to include those exact ideas.

If the retriever is doing its job, the top chunks should mention sales price, for-sale inventory, or months' supply.


In [ ]:
housing_context_text = ''
housing_top_results = []
housing_rank_number = 1

for housing_ranked_result in housing_ranked_results:
    if housing_rank_number <= 3:
        housing_top_results.append(housing_ranked_result)
        housing_context_text = housing_context_text + '[' + housing_ranked_result[1] + ']\n'
        housing_context_text = housing_context_text + housing_ranked_result[2] + '\n'
        housing_context_text = housing_context_text + housing_ranked_result[4] + '\n\n'

    housing_rank_number = housing_rank_number + 1

print('Retrieved context:')
print(housing_context_text)
print()

housing_for_sale_values = []
housing_price_values = []

for housing_top_result in housing_top_results:
    housing_chunk_text = housing_top_result[4]

    if 'New Houses For Sale2:' in housing_chunk_text:
        housing_for_sale_text = housing_chunk_text.split('New Houses For Sale2:')[1]
        housing_for_sale_value = housing_for_sale_text.split()[0]
        housing_for_sale_values.append(housing_for_sale_value)

    if 'Median Sales Price:' in housing_chunk_text:
        housing_price_text = housing_chunk_text.split('Median Sales Price:')[1]
        housing_price_value = housing_price_text.split()[0]
        housing_price_values.append(housing_price_value)

housing_evidence_answer = 'Across the top housing releases, the number of new houses for sale was '
housing_value_index = 0

for housing_for_sale_value in housing_for_sale_values:
    if housing_value_index > 0:
        housing_evidence_answer = housing_evidence_answer + ', '

    housing_evidence_answer = housing_evidence_answer + housing_for_sale_value
    housing_value_index = housing_value_index + 1

housing_evidence_answer = housing_evidence_answer + ', and the median sales price was '
housing_value_index = 0

for housing_price_value in housing_price_values:
    if housing_value_index > 0:
        housing_evidence_answer = housing_evidence_answer + ', '

    housing_evidence_answer = housing_evidence_answer + housing_price_value
    housing_value_index = housing_value_index + 1

housing_evidence_answer = housing_evidence_answer + '.'

print('Evidence-based answer:')
print(housing_evidence_answer)


## Section 3: Labor market corpus

This corpus uses real Federal Reserve speeches that discuss labor market conditions in 2025.

A speech is longer and more open-ended than a short policy statement.
That makes this corpus a good test of whether the same retrieval steps still work on a different style of writing.


In [ ]:
labor_source_names = []
labor_source_names.append('Kugler speech on labor market rebalancing, March 7, 2025')
labor_source_names.append('Powell speech on the economic outlook, April 16, 2025')
labor_source_names.append('Kugler speech on maximum employment, May 9, 2025')

labor_source_urls = []
labor_source_urls.append('https://www.federalreserve.gov/newsevents/speech/kugler20250307a.htm')
labor_source_urls.append('https://www.federalreserve.gov/newsevents/speech/powell20250416a.htm')
labor_source_urls.append('https://www.federalreserve.gov/newsevents/speech/kugler20250509a.htm')

labor_file_paths = []
labor_file_paths.append(corpus_folder / 'labor' / 'kugler_2025_03_07_labor_markets.html')
labor_file_paths.append(corpus_folder / 'labor' / 'powell_2025_04_16_economic_outlook.html')
labor_file_paths.append(corpus_folder / 'labor' / 'kugler_2025_05_09_maximum_employment.html')

labor_documents = []
labor_document_index = 0

for labor_file_path in labor_file_paths:
    labor_html_text = labor_file_path.read_text(encoding='utf-8', errors='ignore')
    labor_soup = BeautifulSoup(labor_html_text, 'html.parser')
    labor_article = labor_soup.find(id='article')
    labor_document_text = ''

    if labor_article is not None:
        for labor_paragraph in labor_article.find_all('p'):
            labor_paragraph_text = labor_paragraph.get_text(' ', strip=True)
            if labor_paragraph_text != '':
                if labor_paragraph_text != 'Share':
                    if 'For media inquiries' not in labor_paragraph_text:
                        if 'Accessible Keys for Video' not in labor_paragraph_text:
                            if 'Watch Live' not in labor_paragraph_text:
                                if 'Return to text' not in labor_paragraph_text:
                                    labor_paragraph_text = labor_paragraph_text.replace(' Share', '')
                                    labor_document_text = labor_document_text + labor_paragraph_text + '\n'

    labor_documents.append(labor_document_text.strip())

    print('Document number:', labor_document_index)
    print('Document name:', labor_source_names[labor_document_index])
    print('Source URL:', labor_source_urls[labor_document_index])
    print('Local file:', labor_file_paths[labor_document_index])
    print('Character count:', len(labor_documents[labor_document_index]))
    print('Preview:')
    print(labor_documents[labor_document_index][0:500])
    print()

    labor_document_index = labor_document_index + 1

labor_question = 'How do these 2025 speeches describe labor market balance and wage pressure?'

print('Question:')
print(labor_question)


## Chunk and retrieve from the labor market corpus

Because these speeches are longer than the earlier documents, chunking matters even more.

A long document often mixes many ideas.
Chunking lets the retriever focus on the smaller parts that talk about labor balance, hiring, or wages.


In [ ]:
labor_chunk_size = common_chunk_size
labor_chunk_overlap = common_chunk_overlap
labor_chunk_records = []
labor_document_index = 0

for labor_document_text in labor_documents:
    labor_start_position = 0
    labor_chunk_number = 0
    labor_text_length = len(labor_document_text)

    while labor_start_position < labor_text_length:
        labor_end_position = labor_start_position + labor_chunk_size

        if labor_end_position > labor_text_length:
            labor_end_position = labor_text_length

        if labor_end_position < labor_text_length:
            labor_search_position = labor_end_position

            while labor_search_position > labor_start_position:
                if labor_document_text[labor_search_position - 1].isspace() is True:
                    labor_end_position = labor_search_position - 1
                    break

                labor_search_position = labor_search_position - 1

        labor_chunk_text = labor_document_text[labor_start_position:labor_end_position].strip()

        if labor_chunk_text != '':
            labor_chunk_record = {
                'source_name': labor_source_names[labor_document_index],
                'source_url': labor_source_urls[labor_document_index],
                'chunk_number': labor_chunk_number,
                'text': labor_chunk_text,
            }
            labor_chunk_records.append(labor_chunk_record)
            labor_chunk_number = labor_chunk_number + 1

        if labor_end_position == labor_text_length:
            labor_next_start_position = labor_text_length
        else:
            labor_next_start_position = labor_end_position - labor_chunk_overlap

            if labor_next_start_position <= labor_start_position:
                labor_next_start_position = labor_end_position

            while labor_next_start_position > 0:
                if labor_document_text[labor_next_start_position - 1].isspace() is True:
                    break

                labor_next_start_position = labor_next_start_position - 1

            while labor_next_start_position < labor_text_length:
                if labor_document_text[labor_next_start_position].isspace() is False:
                    break

                labor_next_start_position = labor_next_start_position + 1

        labor_start_position = labor_next_start_position

    labor_document_index = labor_document_index + 1

labor_chunk_texts = []

for labor_chunk_record in labor_chunk_records:
    labor_chunk_texts.append(labor_chunk_record['text'])

labor_chunk_embeddings = embedding_model.encode(labor_chunk_texts)

labor_question_embedding = embedding_model.encode(labor_question)
labor_similarity_scores = []

for labor_chunk_embedding in labor_chunk_embeddings:
    labor_similarity_value = cosine_similarity([labor_question_embedding], [labor_chunk_embedding])[0][0]
    labor_similarity_scores.append(labor_similarity_value)

labor_chunk_labels = []

for labor_chunk_record in labor_chunk_records:
    labor_chunk_label = labor_chunk_record['source_name'] + '_chunk_' + str(labor_chunk_record['chunk_number'])
    labor_chunk_labels.append(labor_chunk_label)

figure = plt.figure(figsize=(11, 3.5))
plt.bar(labor_chunk_labels, labor_similarity_scores)
plt.title('Labor market retrieval scores')
plt.xlabel('Chunk')
plt.ylabel('Cosine similarity')
plt.xticks(rotation=60, ha='right')
plt.tight_layout()
plt.show()
plt.close(figure)

labor_ranked_results = []
labor_chunk_index = 0

for labor_similarity_value in labor_similarity_scores:
    labor_ranked_result = (
        labor_similarity_value,
        labor_chunk_records[labor_chunk_index]['source_name'],
        labor_chunk_records[labor_chunk_index]['source_url'],
        labor_chunk_records[labor_chunk_index]['chunk_number'],
        labor_chunk_records[labor_chunk_index]['text'],
    )
    labor_ranked_results.append(labor_ranked_result)
    labor_chunk_index = labor_chunk_index + 1

labor_ranked_results = sorted(labor_ranked_results, reverse=True)

labor_rank_number = 1

for labor_ranked_result in labor_ranked_results:
    if labor_rank_number <= 3:
        print('Rank:', labor_rank_number)
        print('Score:', round(labor_ranked_result[0], 4))
        print('Source:', labor_ranked_result[1])
        print('Source URL:', labor_ranked_result[2])
        print('Chunk number:', labor_ranked_result[3])
        print('Chunk text:')
        print(labor_ranked_result[4])
        print()

    labor_rank_number = labor_rank_number + 1


## Build the labor market context and answer

The top labor chunks should now tell us whether these speeches describe the labor market as tight, balanced, cooling, or some mixture of those ideas.

We will again build a short evidence-based answer from the top chunks.


In [ ]:
labor_context_text = ''
labor_top_results = []
labor_rank_number = 1

for labor_ranked_result in labor_ranked_results:
    if labor_rank_number <= 3:
        labor_top_results.append(labor_ranked_result)
        labor_context_text = labor_context_text + '[' + labor_ranked_result[1] + ']\n'
        labor_context_text = labor_context_text + labor_ranked_result[2] + '\n'
        labor_context_text = labor_context_text + labor_ranked_result[4] + '\n\n'

    labor_rank_number = labor_rank_number + 1

print('Retrieved context:')
print(labor_context_text)
print()

labor_answer_keywords = []
labor_answer_keywords.append('labor market')
labor_answer_keywords.append('wage')
labor_answer_keywords.append('earnings')
labor_answer_keywords.append('maximum employment')
labor_answer_keywords.append('stable')

labor_answer_sentences = []

for labor_top_result in labor_top_results:
    labor_chunk_text = labor_top_result[4].replace('\n', ' ')
    labor_sentence_pieces = labor_chunk_text.split('. ')

    for labor_sentence_piece in labor_sentence_pieces:
        labor_sentence = labor_sentence_piece.strip()

        if labor_sentence != '':
            labor_sentence_lower = labor_sentence.lower()

            for labor_answer_keyword in labor_answer_keywords:
                if labor_answer_keyword in labor_sentence_lower:
                    if labor_sentence.endswith('.') is False:
                        labor_sentence = labor_sentence + '.'

                    if labor_sentence not in labor_answer_sentences:
                        labor_answer_sentences.append(labor_sentence)

                    break

labor_evidence_answer = ''

for labor_sentence in labor_answer_sentences:
    if labor_sentence not in labor_evidence_answer:
        if labor_evidence_answer != '':
            labor_evidence_answer = labor_evidence_answer + ' '
        labor_evidence_answer = labor_evidence_answer + labor_sentence

print('Evidence-based answer:')
print(labor_evidence_answer)


## Compare the three corpus results

The final comparison asks one simple question.

When we hold the retrieval method constant, how does the strongest evidence differ across three different economics corpora?


In [ ]:
corpus_names = []
corpus_names.append('Monetary policy')
corpus_names.append('Housing market')
corpus_names.append('Labor market')

top_source_names = []
top_source_names.append(monetary_top_results[0][1])
top_source_names.append(housing_top_results[0][1])
top_source_names.append(labor_top_results[0][1])

top_source_urls = []
top_source_urls.append(monetary_top_results[0][2])
top_source_urls.append(housing_top_results[0][2])
top_source_urls.append(labor_top_results[0][2])

top_scores = []
top_scores.append(monetary_top_results[0][0])
top_scores.append(housing_top_results[0][0])
top_scores.append(labor_top_results[0][0])

corpus_index = 0

for corpus_name in corpus_names:
    print('Corpus:', corpus_name)
    print('Top source:', top_source_names[corpus_index])
    print('Top source URL:', top_source_urls[corpus_index])
    print('Top score:', round(top_scores[corpus_index], 4))
    print()
    corpus_index = corpus_index + 1

figure = plt.figure(figsize=(8, 3.5))
plt.bar(corpus_names, top_scores)
plt.title('Top retrieval score in each real corpus')
plt.ylabel('Cosine similarity')
plt.tight_layout()
plt.show()
plt.close(figure)


## Summary

In this notebook, you ran the same retrieval workflow on three real economics corpora.

You loaded official source files, split the text into chunks, embedded those chunks, and retrieved the passages that best matched each question.
You also saw that the same RAG structure can work across HTML pages, PDF reports, and longer speech texts.

That is the main lesson: the retrieval method stayed the same, but the real corpus changed.

### References

Monetary policy corpus:
- https://www.federalreserve.gov/newsevents/pressreleases/monetary20250129a.htm
- https://www.federalreserve.gov/newsevents/pressreleases/monetary20250319a.htm
- https://www.federalreserve.gov/newsevents/pressreleases/monetary20250507a.htm

Housing market corpus:
- https://www.census.gov/construction/nrs/pdf/newressales_202501.pdf
- https://www.census.gov/construction/nrs/pdf/newressales_202502.pdf
- https://www.census.gov/construction/nrs/pdf/newressales_202503.pdf

Labor market corpus:
- https://www.federalreserve.gov/newsevents/speech/kugler20250307a.htm
- https://www.federalreserve.gov/newsevents/speech/powell20250416a.htm
- https://www.federalreserve.gov/newsevents/speech/kugler20250509a.htm
